# GA Feature Selector: Evolutionary Search in ~30 Lines

## What You Will Do
You will implement a complete genetic algorithm for feature selection from scratch — no DEAP, no library beyond NumPy and scikit-learn. By the end you will have a working evolutionary search that outperforms a simple filter baseline, and you will understand every design decision: chromosome encoding, selection pressure, crossover, and mutation.

## Why Genetic Algorithms for Feature Selection?
Feature selection is a combinatorial search problem. For 30 features there are $2^{30} \approx 10^9$ possible subsets. Exhaustive search is impossible. A GA explores this space intelligently by evolving a population of candidate subsets, applying selection pressure toward high-accuracy subsets, and using crossover to combine good partial solutions from different individuals.

## The 30-Line Implementation
The core algorithm fits in 30 lines. Every single line has a job — we will look at each one.

## Estimated Time: under 2 minutes

---

## Setup

In [ ]:
# Purpose: Import all dependencies
# Key Concept: The entire GA needs only numpy + sklearn — no evolutionary computation library

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.feature_selection import mutual_info_classif

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Setup complete.")

## Load the Dataset

In [ ]:
# Purpose: Load the breast cancer dataset
# We use a Decision Tree as the fitness evaluator (fast, ~1ms per CV fold)

data = load_breast_cancer(as_frame=True)
X = data.data.values
y = data.target.values
feature_names = data.feature_names.tolist()
N_FEATURES = X.shape[1]

print(f"Samples : {X.shape[0]}")
print(f"Features: {N_FEATURES}")
print(f"Problem : binary classification (malignant / benign)")

## The GA Design

Before code, the design decisions:

| Component | Choice | Why |
|---|---|---|
| **Chromosome** | Binary vector of length `n_features` | 1 = include, 0 = exclude. Simplest possible encoding. |
| **Fitness** | 3-fold CV accuracy of a Decision Tree | Fast to evaluate; penalises subsets that overfit |
| **Selection** | Tournament selection (k=3) | Simple, tunable selection pressure, O(1) per individual |
| **Crossover** | Uniform crossover (p=0.5 per bit) | Lets good bits from both parents combine freely |
| **Mutation** | Bit-flip with p=1/n_features | Ensures every gene can mutate; rate scales with genome size |
| **Elitism** | Top 2 individuals survive unchanged | Prevents regression from one generation to the next |

## The Core GA: ~30 Lines

In [ ]:
# Purpose: Implement a complete GA feature selector from scratch
# Key Concept: Each of the 5 components (init, fitness, selection, crossover, mutation)
#              is a single short function — the full loop runs each generation

def init_population(pop_size, n_features, rng):
    """Random binary chromosomes. Each bit = include/exclude one feature."""
    return rng.randint(0, 2, size=(pop_size, n_features))


def fitness(chromosome, X, y, model, cv):
    """3-fold CV accuracy of model trained on features indicated by chromosome.
    Returns 0 if no features are selected (degenerate chromosome)."""
    idx = np.where(chromosome)[0]
    if len(idx) == 0:
        return 0.0
    return cross_val_score(model, X[:, idx], y, cv=cv, scoring='accuracy').mean()


def tournament_select(population, fitnesses, k=3, rng=None):
    """Run k-way tournament. Pick k random individuals; return the best one."""
    candidates = rng.choice(len(population), size=k, replace=False)
    winner = candidates[np.argmax(fitnesses[candidates])]
    return population[winner].copy()


def uniform_crossover(parent_a, parent_b, rng):
    """Each bit is independently drawn from either parent with equal probability."""
    mask = rng.randint(0, 2, size=len(parent_a)).astype(bool)
    child = np.where(mask, parent_a, parent_b)
    return child


def mutate(chromosome, mutation_rate, rng):
    """Flip each bit with probability mutation_rate. Operates in-place."""
    flip_mask = rng.random(len(chromosome)) < mutation_rate
    chromosome[flip_mask] ^= 1
    return chromosome


def run_ga(
    X, y,
    pop_size=15, n_generations=8,
    mutation_rate=None, tournament_k=3,
    n_elites=2, random_state=42
):
    """
    Run genetic algorithm feature selection.

    Parameters
    ----------
    X : ndarray (n_samples, n_features)
    y : ndarray (n_samples,)
    pop_size : int  — number of candidate subsets
    n_generations : int
    mutation_rate : float or None  — defaults to 1/n_features
    tournament_k : int  — tournament size (selection pressure)
    n_elites : int  — individuals carried over unchanged
    random_state : int

    Returns
    -------
    best_chromosome : ndarray of bool
    history : list of (best_fitness, mean_fitness) per generation
    """
    rng = np.random.RandomState(random_state)
    n_features = X.shape[1]
    if mutation_rate is None:
        mutation_rate = 1.0 / n_features  # classic setting

    model = DecisionTreeClassifier(random_state=random_state)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)

    # --- Initialise ---
    population = init_population(pop_size, n_features, rng)
    history = []

    for gen in range(n_generations):
        # --- Evaluate ---
        fitnesses = np.array([fitness(ind, X, y, model, cv) for ind in population])

        best_idx = np.argmax(fitnesses)
        history.append((fitnesses[best_idx], fitnesses.mean()))
        print(f"  Gen {gen+1:2d}: best={fitnesses[best_idx]:.4f}  "
              f"mean={fitnesses.mean():.4f}  "
              f"features={population[best_idx].sum()}")

        # --- Elitism: keep top n_elites unchanged ---
        elite_idx = np.argsort(fitnesses)[-n_elites:]
        new_population = [population[i].copy() for i in elite_idx]

        # --- Fill rest of new population via selection + crossover + mutation ---
        while len(new_population) < pop_size:
            parent_a = tournament_select(population, fitnesses, tournament_k, rng)
            parent_b = tournament_select(population, fitnesses, tournament_k, rng)
            child = uniform_crossover(parent_a, parent_b, rng)
            child = mutate(child, mutation_rate, rng)
            new_population.append(child)

        population = np.array(new_population)

    # Final evaluation
    fitnesses = np.array([fitness(ind, X, y, model, cv) for ind in population])
    best_chromosome = population[np.argmax(fitnesses)]
    return best_chromosome, history


print("GA functions defined. Ready to run.")
print(f"Chromosome length: {N_FEATURES} bits (one per feature)")
print(f"Search space: 2^{N_FEATURES} = {2**N_FEATURES:,} possible subsets")

## Run the GA

We run 8 generations with a population of 15 — 120 total fitness evaluations. Each evaluation is a 3-fold CV on a Decision Tree, which completes in milliseconds. Total runtime: under 15 seconds.

In [ ]:
# Purpose: Execute the GA and recover the best feature subset
# Key Concept: 15 individuals x 8 generations x 3-fold CV is 360 model fits —
#              a tiny fraction of the 2^30 = 1 billion possible subsets evaluated

print("Running GA feature selection...")
print(f"Population: 15 | Generations: 8 | CV: 3-fold | Model: Decision Tree")
print()

best_chromosome, history = run_ga(
    X, y,
    pop_size=15,
    n_generations=8,
    tournament_k=3,
    n_elites=2,
    random_state=RANDOM_STATE
)

ga_selected_idx = np.where(best_chromosome)[0]
ga_selected = [feature_names[i] for i in ga_selected_idx]

print(f"\nGA selected {len(ga_selected)} features:")
for name in ga_selected:
    print(f"  - {name}")

## Filter Baseline: Mutual Information Top-K

We compare the GA against the simplest baseline: select the top-K features by mutual information, where K equals the number of features the GA chose. This is a fair comparison because we give the filter the same budget.

In [ ]:
# Purpose: Build MI filter baseline with the same number of features as GA
# Key Concept: Comparing against the same feature count makes the evaluation fair;
#              the GA is only valuable if it beats a simpler method of equal size

K = len(ga_selected)  # Use same budget as GA

mi_scores = mutual_info_classif(X, y, random_state=RANDOM_STATE)
mi_top_idx = np.argsort(mi_scores)[::-1][:K]
mi_selected = [feature_names[i] for i in mi_top_idx]

print(f"MI filter top-{K} features:")
for name in mi_selected:
    print(f"  - {name}")

overlap = set(ga_selected) & set(mi_selected)
print(f"\nOverlap with GA: {len(overlap)} / {K} features")
print("Features both methods agree on:", sorted(overlap))

## Performance Comparison

We evaluate both methods using the same Decision Tree with 5-fold stratified CV. The GA optimised for 3-fold CV during search — using 5-fold here is a stricter out-of-search-loop evaluation.

In [ ]:
# Purpose: Compare GA vs MI baseline vs all-features on 5-fold CV accuracy
# Key Concept: The evaluation CV (5-fold) is separate from the fitness CV (3-fold)
#              to prevent the GA from overfitting to the fitness landscape

dt_eval = DecisionTreeClassifier(random_state=RANDOM_STATE)
cv_eval = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scores_all = cross_val_score(dt_eval, X, y, cv=cv_eval, scoring='accuracy')
scores_mi = cross_val_score(dt_eval, X[:, mi_top_idx], y, cv=cv_eval, scoring='accuracy')
scores_ga = cross_val_score(dt_eval, X[:, ga_selected_idx], y, cv=cv_eval, scoring='accuracy')

print(f"{'Method':<30} {'Mean Acc':>9} {'Std':>7} {'Features':>10}")
print('-' * 60)
print(f"{'All features':<30} {scores_all.mean():>9.4f} {scores_all.std():>7.4f} "
      f"{X.shape[1]:>10}")
print(f"{'MI filter (top-K)':<30} {scores_mi.mean():>9.4f} {scores_mi.std():>7.4f} "
      f"{K:>10}")
print(f"{'GA selected':<30} {scores_ga.mean():>9.4f} {scores_ga.std():>7.4f} "
      f"{len(ga_selected_idx):>10}")

ga_vs_all = scores_ga.mean() - scores_all.mean()
ga_vs_mi = scores_ga.mean() - scores_mi.mean()
print(f"\nGA vs all features: {ga_vs_all:+.4f}")
print(f"GA vs MI baseline : {ga_vs_mi:+.4f}")

## Visualisation — Fitness Improvement Over Generations

In [ ]:
# Purpose: Plot fitness convergence and performance comparison
# Key Concept: The convergence curve reveals whether the GA found a good solution
#              quickly or is still searching — flat curves suggest premature convergence

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: GA convergence curve
ax = axes[0]
generations = range(1, len(history) + 1)
best_per_gen = [h[0] for h in history]
mean_per_gen = [h[1] for h in history]

ax.plot(generations, best_per_gen, 'o-', color='#d62728', linewidth=2.5,
        markersize=7, label='Best individual')
ax.plot(generations, mean_per_gen, 's--', color='#1f77b4', linewidth=1.5,
        markersize=5, alpha=0.7, label='Population mean')
ax.fill_between(generations, mean_per_gen, best_per_gen, alpha=0.15, color='#d62728')
ax.set_xlabel('Generation', fontsize=11)
ax.set_ylabel('3-Fold CV Accuracy (fitness)', fontsize=11)
ax.set_title('GA Fitness Convergence\n'
             '(gap between best and mean shows selection pressure)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xticks(list(generations))
ax.set_ylim(max(0.5, min(mean_per_gen) - 0.05), 1.02)

# Right: Method comparison bar chart
ax2 = axes[1]
labels = [f'All\n({X.shape[1]} feat)', f'MI filter\n(top {K})', f'GA\n({len(ga_selected)} feat)']
means = [scores_all.mean(), scores_mi.mean(), scores_ga.mean()]
stds = [scores_all.std(), scores_mi.std(), scores_ga.std()]
colors = ['#7f7f7f', '#ff7f0e', '#2ca02c']

bars = ax2.bar(labels, means, yerr=stds, capsize=6, color=colors, alpha=0.85, edgecolor='white')
ax2.set_ylabel('5-Fold CV Accuracy', fontsize=11)
ax2.set_title('GA vs Baseline: 5-Fold CV Accuracy\n'
              f'(evaluation uses separate CV from GA fitness)', fontsize=11)
ax2.set_ylim(0.85, 1.0)
ax2.grid(axis='y', alpha=0.3)
# Add value labels on bars
for bar, mean, std in zip(bars, means, stds):
    ax2.text(bar.get_x() + bar.get_width()/2, mean + std + 0.002,
             f'{mean:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## What Each Line Actually Does

Let us annotate the core GA loop — 30 lines, every one with a purpose.

In [ ]:
# Purpose: Walk through the annotated GA loop so the algorithm is fully transparent
# Key Concept: Understanding each step enables you to modify the GA for new problems

annotated_ga = """
# --- GENETIC ALGORITHM: ANNOTATED CORE LOOP (30 lines) ---

rng = np.random.RandomState(random_state)       # 1. Reproducible random number generator
mutation_rate = 1.0 / n_features                # 2. Classic rate: expect ~1 bit flip per chromosome
population = init_population(pop_size, ...)     # 3. Random binary chromosomes — diverse start

for gen in range(n_generations):                # 4. Outer evolution loop

    fitnesses = [fitness(ind, ...) for ind]     # 5. Evaluate every individual (the expensive step)

    best_idx = np.argmax(fitnesses)             # 6. Find this generation's best individual
    history.append((fitnesses[best_idx], ...))  # 7. Record progress for convergence plot

    elite_idx = np.argsort(fitnesses)[-n_elites:] # 8. Identify the n_elites best individuals
    new_pop = [population[i].copy() for i in elite_idx]  # 9. Copy elites unchanged

    while len(new_pop) < pop_size:              # 10. Fill remaining slots

        parent_a = tournament_select(...)       # 11. Pick winner of k-way tournament from population
        parent_b = tournament_select(...)       # 12. Pick a second (independent) winner

        child = uniform_crossover(a, b, rng)    # 13. Each bit inherited from a or b with 50% probability
        child = mutate(child, rate, rng)        # 14. Randomly flip bits to maintain diversity

        new_pop.append(child)                   # 15. Add to next generation

    population = np.array(new_pop)             # 16. Replace population — generational model

# After all generations:
fitnesses = [fitness(ind, ...) for ind]        # 17. Final evaluation
best = population[np.argmax(fitnesses)]        # 18. Return the best chromosome found
"""
print(annotated_ga)

## Exercise: Tune Population Size and Tournament Size

**Task:** Rerun the GA with `pop_size=30` and `tournament_k=5`. Compare the convergence curve (does it find a good solution faster?) and the final 5-fold accuracy.

A larger population explores more of the search space per generation. A larger tournament k applies more selection pressure — the best individual wins more often, which speeds convergence but risks premature convergence to a local optimum.

<details>
<summary>Hint 1</summary>
Call run_ga(X, y, pop_size=30, n_generations=8, tournament_k=5, random_state=42) and store the result as best_chromosome_tuned, history_tuned.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
After running, compare: (1) convergence speed — does the best fitness reach its peak earlier? (2) final subset size — does larger pop find a smaller or larger subset? (3) accuracy — cross_val_score with the tuned chromosome.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Run the GA with tuned parameters

POP_SIZE_TUNED = 30       # Try: 20, 30, 40
TOURNAMENT_K_TUNED = 5   # Try: 2, 5, 8

print(f"Running tuned GA: pop_size={POP_SIZE_TUNED}, tournament_k={TOURNAMENT_K_TUNED}")
print()

best_chromosome_tuned, history_tuned = run_ga(
    X, y,
    pop_size=POP_SIZE_TUNED,
    n_generations=8,
    tournament_k=TOURNAMENT_K_TUNED,
    n_elites=2,
    random_state=RANDOM_STATE
)

ga_tuned_idx = np.where(best_chromosome_tuned)[0]
scores_ga_tuned = cross_val_score(
    dt_eval, X[:, ga_tuned_idx], y, cv=cv_eval, scoring='accuracy'
)

print(f"\nTuned GA: {len(ga_tuned_idx)} features, accuracy = {scores_ga_tuned.mean():.4f}")
print(f"Original GA: {len(ga_selected_idx)} features, accuracy = {scores_ga.mean():.4f}")

your_tuned_accuracy = scores_ga_tuned.mean()

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise():
    assert best_chromosome_tuned is not None, "Run the tuned GA above."
    assert len(best_chromosome_tuned) == N_FEATURES, \
        f"Chromosome length should be {N_FEATURES}; got {len(best_chromosome_tuned)}."
    assert your_tuned_accuracy > 0.88, (
        f"Tuned GA accuracy {your_tuned_accuracy:.4f} seems low. "
        "Check that the GA ran for 8 generations with pop_size >= 20."
    )
    assert len(history_tuned) == 8, \
        "history_tuned should have one entry per generation (8 entries)."
    print(f"Exercise passed! Tuned GA: {len(ga_tuned_idx)} features, "
          f"accuracy = {your_tuned_accuracy:.4f}.")

test_exercise()

## Summary

**Key Takeaways:**

1. **Binary chromosome encoding** maps the feature selection problem onto a string of bits — the natural representation for include/exclude decisions.
2. **Tournament selection** provides tunable selection pressure: larger k converges faster but risks losing diversity.
3. **Uniform crossover** allows any combination of parent bits — effective when good features are spread across individuals.
4. **Mutation rate = 1/n_features** ensures expected one flip per chromosome — the empirically validated default.
5. **Elitism** prevents regression by preserving the best individuals across generations.
6. A GA with 15 individuals and 8 generations performs 360 model evaluations — a tiny fraction of the $2^{30}$ search space.

**What is next:**
- `causal_vs_predictive_features.ipynb` — when predictive features fail under distribution shift
- Module 5 (Genetic Algorithms) for multi-objective GA and niching strategies
- Module 6 (Evolutionary and Swarm) for PSO and differential evolution alternatives